# Работа во время учёбы: цена, польза и выбор студентов

Анализ задачи хакатона «Совмещение учёбы и работы: польза или вред?» на данных лонгитюдного исследования «Модели образовательного поведения студентов».

**Вопрос:** как занятость, её интенсивность и соответствие работы специальности связаны с учебным опытом, успеваемостью и самочувствием студентов?

**Важно:** это наблюдательный анализ. Результаты показывают связи и не доказывают, что работа сама по себе вызывает изменения. Панельное сравнение ограничено выбыванием участников и неполными ответами. Гипотезы сформулированы после первичного изучения данных.

## Запуск
1. Получите разрешённую к использованию копию массива в формате `.sav` у организаторов исследования. Исходные данные не входят в репозиторий.
2. Сохраните файл как `data/private/longitudinal.sav` либо укажите путь через переменную окружения `LONGITUDINAL_DATA`.
3. Установите зависимости из `requirements.txt` и запустите все ячейки.

Ноутбук сохраняет графики в `figs/` и сводную таблицу в `results_D.csv`. Не публикуйте эти производные файлы до проверки правил распространения результатов исследования.

## План анализа
- описать выборку, занятость в двух волнах и выбытие;
- сравнить планируемый бюджет времени и учебные/здоровьесберегающие исходы;
- оценить, как воспринимаемая польза и трудности соотносятся с часами, мотивом и соответствием специальности;
- изучить переходы в занятости и факторы, наблюдавшиеся до начала работы;
- проверить чувствительность выводов к альтернативным определениям и множественным проверкам.


## 1. Загрузка, чистка, производные переменные

In [ ]:
# Установите зависимости заранее: pip install -r requirements.txt


In [ ]:
import os, glob, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import pyreadstat, patsy
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 200)
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
os.makedirs("figs", exist_ok=True)
def save(name):
    plt.tight_layout(); plt.savefig(f"figs/{name}.png", dpi=200, bbox_inches="tight"); plt.show()

DATA_PATH = os.environ.get("LONGITUDINAL_DATA", "data/private/longitudinal.sav")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Не найден массив: {DATA_PATH}. Получите разрешённую копию данных и "
        "положите её по этому пути или задайте LONGITUDINAL_DATA."
    )
df, meta = pyreadstat.read_sav(DATA_PATH)
VL = meta.variable_value_labels
print("Загружен массив:", os.path.basename(DATA_PATH), df.shape)

In [ ]:
d = df.copy()
# --- строки -> числа
for c in ["wave3", "dropout1_3w", "mark_3w"]:
    d[c] = pd.to_numeric(d[c].astype(str).str.strip().replace({"": np.nan, "nan": np.nan}), errors="coerce")
# --- коды 88/99 -> NaN (кроме часов и процентов времени)
skip = {"work_hours", "work_hours_4w"} | {c for c in d.columns if c.startswith("hours_3w")}
text_cols = [c for c in d.columns if not pd.api.types.is_numeric_dtype(d[c])]
cat_cols = [c for c in d.columns if c not in skip and c not in text_cols]
d[cat_cols] = d[cat_cols].replace({88: np.nan, 99: np.nan})
for c in ["satisfaction_3w", "satisfaction_4w", "Work_major_4w"]:
    d.loc[d[c] == 5, c] = np.nan
for c in ["work_hours", "work_hours_4w"]:
    d.loc[(d[c] < 0) | (d[c] > 100), c] = np.nan

d["in3"] = d.wave3 == 1
d["in4"] = d.wave4 == 1

In [ ]:
# ===== Производные переменные =====
def bin_(s, cond):          # бинарка с сохранением NaN
    return np.where(s.isna(), np.nan, cond.astype(float))
def cat_na(s, mapper):      # категории как ASCII-коды, пропуск -> 'na'
    return s.map(mapper).fillna("na")

# --- ковариаты (ASCII-метки: так проще в формулах)
d["sex_c"]     = cat_na(d.sex, {1: "m", 2: "f"})
d["funding_c"] = cat_na(d.funding, {1: "budget", 2: "paid", 3: "target"})
d["major_c"]   = cat_na(d.bigmajor, {1: "math", 2: "eng", 5: "soc", 7: "hum", 3: "oth", 4: "oth", 6: "oth", 8: "oth"})
d["ses_c"]     = cat_na(d.ses1, {1: "low", 2: "low", 3: "low", 4: "mid", 5: "high", 6: "vhigh"})
CTRL = "C(sex_c) + C(funding_c, Treatment('budget')) + C(major_c, Treatment('eng')) + C(ses_c, Treatment('high'))"

# --- занятость
WT = {1: "full", 2: "part", 3: "casual", 4: "none"}
for w, c in [("3", "work1"), ("4", "work1_4w")]:
    d[f"wt{w}"]  = d[c].map(WT)                                    # тип занятости
    d[f"emp{w}"] = np.where(d[c].isna(), np.nan, d[c].isin([1, 2, 3]).astype(float))
d["has3"], d["has4"] = d.emp3.notna(), d.emp4.notna()
d["panel"] = d.has3 & d.has4

def tr(r):
    if np.isnan(r.emp3) or np.isnan(r.emp4): return np.nan
    return {(0, 0): "stay_no", (0, 1): "start", (1, 0): "quit", (1, 1): "stay_yes"}[(r.emp3, r.emp4)]
d["trans"] = d.apply(tr, axis=1)

# --- интенсивность (часы за прошлую неделю), категории только у работающих; не работает = 'none'
def hcat(h, emp):
    out = pd.cut(h, [-0.1, 10, 20, 100], labels=["le10", "h11_20", "h21p"]).astype(object)
    out = pd.Series(out, index=h.index)
    out[emp == 0] = "none"
    out[(emp == 1) & h.isna()] = np.nan
    return out
d["hc3"] = hcat(d.work_hours,    d.emp3)
d["hc4"] = hcat(d.work_hours_4w, d.emp4)

# --- тип работы (4 волна): соответствие специальности и мотив
d["match4"] = np.where(d.emp4 == 0, "none", d.Work_major_4w.map({1: "low", 2: "low", 3: "high", 4: "high"}))
d.loc[(d.emp4 == 1) & d.Work_major_4w.isna(), "match4"] = np.nan
d["match4"] = d["match4"].replace("nan", np.nan)
mot = {1: "money", 2: "money", 3: "money", 4: "exp", 5: "exp"}
d["motive4"] = np.where(d.emp4 == 0, "none", d.work4_4w.map(mot))
d.loc[(d.emp4 == 1) & d.work4_4w.map(mot).isna(), "motive4"] = np.nan
d["motive4"] = d["motive4"].replace("nan", np.nan)
d["motive3"] = np.where(d.emp3 == 0, "none", d.work4.map(mot))
d.loc[(d.emp3 == 1) & d.work4.map(mot).isna(), "motive3"] = np.nan
d["motive3"] = d["motive3"].replace("nan", np.nan)

# --- сон
bed  = lambda c: (19 + c) % 24
wake = lambda c: 2 + c
for w in ["3w", "4w"]:
    wd = (wake(d[f"sleep3_{w}"]) - bed(d[f"sleep1_{w}"])) % 24
    we = (wake(d[f"sleep4_{w}"]) - bed(d[f"sleep2_{w}"])) % 24
    d[f"sleep_avg_{w}"] = (5 * wd + 2 * we) / 7
    mwd = (bed(d[f"sleep1_{w}"]) + wd / 2) % 24
    mwe = (bed(d[f"sleep2_{w}"]) + we / 2) % 24
    df_ = (mwe - mwd).abs()
    d[f"sjl_{w}"] = np.minimum(df_, 24 - df_)
    d[f"short_sleep_{w}"] = bin_(d[f"sleep_avg_{w}"], d[f"sleep_avg_{w}"] < 7)      # <7 часов в среднем

# --- исходы
for w in ["3w", "4w"]:
    d[f"mark_hi_{w}"]  = bin_(d[f"mark_{w}"], d[f"mark_{w}"] >= 3)
    d[f"mark_top_{w}"] = bin_(d[f"mark_{w}"], d[f"mark_{w}"] == 4)
    d[f"drop_{w}"]     = bin_(d[f"dropout1_{w}"], d[f"dropout1_{w}"] >= 2)
    d[f"sat_hi_{w}"]   = bin_(d[f"satisfaction_{w}"], d[f"satisfaction_{w}"] >= 3)
    d[f"again_{w}"]    = bin_(d[f"edu3_{w}"], d[f"edu3_{w}"] == 1)
    d[f"smoke_{w}"]    = np.where(d[f"smoking_{w}"].isin([1, 2]), (d[f"smoking_{w}"] == 1).astype(float), np.nan)
    d[f"alco_{w}"]     = np.where(d[f"alco_{w}"].isin([1, 2]), (d[f"alco_{w}"] == 1).astype(float), np.nan)
d["sport_3w"] = bin_(d.pe_3w_1, d.pe_3w_1 == 1)
d["sport_4w"] = bin_(d.pe_4w,   d.pe_4w == 1)

# --- самооценка совмещения (4 волна, только работающие): согласен = 3-4
for k, nm in [(1, "self_help"), (2, "self_hurt"), (3, "self_hard")]:
    s = d[f"intern5_4w_{k}"]
    d[nm] = bin_(s, s >= 3)
    d[nm + "_ord"] = s
# причины мыслей об отчислении, связанные с работой/деньгами (только думавшие об отчислении)
d["r_work_3w"] = np.where(d.drop_3w == 1, d["dropout2_3w_1_7"], np.nan)
d["r_econ_3w"] = np.where(d.drop_3w == 1, d["dropout2_3w_1_6"], np.nan)
d["r_work_4w"] = d["dropout2_4w_7"]
d["r_econ_4w"] = d["dropout2_4w_6"]
print("Готово. Столбцов:", d.shape[1])

In [ ]:
# --- Хелперы для моделей ---
RES = []   # единая таблица результатов

def _prep(formula, data):
    y, X = patsy.dmatrices(formula, data, return_type="dataframe")
    return y, X

def fit(formula, data, kind, tag, outcome, keep, w=None):
    # Подгоняет модель и сохраняет интересующие коэффициенты (keep: подстрока имени терма) в RES
    try:
        if kind == "logit":
            m = smf.logit(formula, data).fit(disp=0, cov_type="HC1", maxiter=200)
        elif kind == "ols":
            m = (smf.wls(formula, data, weights=w) if w is not None else smf.ols(formula, data)).fit(cov_type="HC1")
        elif kind == "wlogit":
            m = smf.glm(formula, data, family=sm.families.Binomial(), freq_weights=w).fit(cov_type="HC1")
        elif kind == "ordered":
            y, X = _prep(formula, data); X = X.drop(columns="Intercept")
            m = OrderedModel(y.iloc[:, 0], X, distr="logit").fit(method="bfgs", disp=0, maxiter=500)
        else:
            raise ValueError(kind)
    except Exception as e:
        print(f"  [!] {tag}/{outcome}: {type(e).__name__}: {str(e)[:80]}"); return None
    n = int(m.nobs)
    ci = m.conf_int()
    out = []
    for term in m.params.index:
        if any(k in term for k in keep) and "/" not in term:
            b = m.params[term]; lo, hi = ci.loc[term]
            r = dict(tag=tag, outcome=outcome, term=term, coef=b, lo=lo, hi=hi, p=m.pvalues[term], n=n, kind=kind)
            if kind in ("logit", "wlogit", "ordered"):
                r.update(OR=np.exp(b), OR_lo=np.exp(lo), OR_hi=np.exp(hi))
            out.append(r)
    RES.extend(out)
    return m

def show(tag, outcome=None):
    t = pd.DataFrame(RES); t = t[t.tag == tag]
    if outcome: t = t[t.outcome == outcome]
    cols = ["outcome", "term", "n", "coef", "OR", "OR_lo", "OR_hi", "p"]
    return t[[c for c in cols if c in t.columns]].round(3)

def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

In [ ]:
# Дополнительные переменные для вопроса D
d["intent"]    = np.where(d.work_intentions.isna(), np.nan, (d.work_intentions == 1).astype(float))   # «планирую искать работу» (3в, неработающие)
d["intern_3w"] = np.where(d.intern1.isna(), np.nan, (d.intern1 == 1).astype(float))
for k in range(1, 7): d[f"plan{k}"] = d[f"hours_3w_{k}"]     # планируемые % времени (3 волна)
bad_sum = ((d[[f"plan{k}" for k in range(1, 7)]].sum(axis=1) != 100) & d.plan1.notna()).sum()
print("Планируемое время: строк, где сумма не 100:", bad_sum, "(исключаем)")
d.loc[(d[[f"plan{k}" for k in range(1, 7)]].sum(axis=1) != 100), [f"plan{k}" for k in range(1, 7)]] = np.nan

# Работающие 4-й волны с двумя ключевыми измерениями
w4 = d[d.emp4 == 1].copy()
w4["low_match"] = np.where(w4.match4.isin(["low", "high"]), (w4.match4 == "low").astype(float), np.nan)
w4["h21"]       = np.where(w4.work_hours_4w.notna(), (w4.work_hours_4w >= 21).astype(float), np.nan)
w4["exp_mot"]   = np.where(w4.motive4.isin(["money", "exp"]), (w4.motive4 == "exp").astype(float), np.nan)
prof = {(0, 0): "Карьерный старт (по спец., ≤20 ч)", (0, 1): "Карьерный марафон (по спец., 21+ ч)",
        (1, 0): "Подработка (не по спец., ≤20 ч)",   (1, 1): "Вторая смена (не по спец., 21+ ч)"}
w4["profile"] = [prof.get((a, b)) if not (np.isnan(a) or np.isnan(b)) else np.nan for a, b in zip(w4.low_match, w4.h21)]
print("Работающих (4в):", len(w4), "| с профилем:", w4.profile.notna().sum())
print(w4.profile.value_counts())
RES2 = []
def lpm(y, term, data, keep, scale=100):
    # линейная вероятностная модель с робастными SE; коэффициент = разница в п.п. (scale=100) или в минутах (60)
    m = smf.ols(f"{y} ~ {term} + {CTRL}", data).fit(cov_type="HC1")
    b, (lo, hi), p = m.params[keep], m.conf_int().loc[keep], m.pvalues[keep]
    return dict(diff=b * scale, lo=lo * scale, hi=hi * scale, p=p, n=int(m.nobs))

## 1b. Данные и выборка (слайд 3)
Воронка: от всех строк файла до анализируемой панели, плюс проверка осыпания между волнами.

In [ ]:
P = d[d.panel]
steps = [("Строк в файле", len(d)),
         ("Участвовали в 3-й или 4-й волне", int((d.in3 | d.in4).sum())),
         ("Участвовали в обеих волнах", int((d.in3 & d.in4).sum())),
         ("Ответили о занятости в обеих волнах (панель)", int(d.panel.sum()))]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
mx = steps[0][1]
for i, (nm, v) in enumerate(steps):
    ax[0].barh(len(steps) - 1 - i, v, color="#4C78A8", alpha=1 - .18 * i, left=(mx - v) / 2)
    ax[0].text(mx / 2, len(steps) - 1 - i, f"{v:,}".replace(",", " "), ha="center", va="center", color="white", fontweight="bold")
ax[0].set_yticks(range(len(steps))); ax[0].set_yticklabels([s[0] for s in steps][::-1]); ax[0].set_xticks([]); ax[0].set_title("Воронка выборки")
tr_ = P.trans.value_counts().reindex(["stay_no", "start", "quit", "stay_yes"])
lab_ = {"stay_no": "не работали\nв обеих", "start": "начали\nработать", "quit": "бросили\nработу", "stay_yes": "работали\nв обеих"}
ax[1].bar([lab_[k] for k in tr_.index], tr_.values, color=["#BBBBBB", "#54A24B", "#E45756", "#4C78A8"])
for i, v in enumerate(tr_.values): ax[1].text(i, v, int(v), ha="center", va="bottom")
ax[1].set_title("Панель: переходы 3 → 4 волна")
save("D00_funnel")
print("Работа в 3в у участников 3 волны:", int(d.has3.sum()), "| в 4в у участников 4 волны:", int(d.has4.sum()))
print("Не заполнен блок о работе в 4в среди тех, кто в обеих волнах:", int((d.in3 & d.in4 & ~d.has4).sum()))

In [ ]:
# Осыпание: кто остаётся в панели (среди участников 3-й волны с данными о занятости)
A3 = d[d.has3].copy(); A3["kept"] = A3.panel.astype(float)
print(f"Доля оставшихся в панели: {A3.kept.mean()*100:.1f}% (n={len(A3)})\n")
rows = []
for var, nm in [("wt3", "тип занятости 3в"), ("funding_c", "финансирование"), ("sex_c", "пол"), ("major_c", "направление")]:
    g = A3.groupby(var)["kept"].agg(["mean", "count"])
    chi = stats.chi2_contingency(pd.crosstab(A3[var], A3.kept))[1]
    for k, r in g.iterrows(): rows.append({"признак": nm, "группа": k, "осталось, %": round(r["mean"] * 100, 1), "n": int(r["count"]), "p (хи²)": round(chi, 3)})
print(pd.DataFrame(rows).to_string(index=False))
for v, nm in [("mark_hi_3w", "оценки 4–5"), ("sat_hi_3w", "удовлетворён"), ("drop_3w", "думал об отчислении")]:
    a = A3.loc[A3.kept == 1, v].dropna(); b = A3.loc[A3.kept == 0, v].dropna()
    print(f"{nm}: остались {a.mean():.2f} vs выбыли {b.mean():.2f}, p={stats.ttest_ind(a, b, equal_var=False).pvalue:.3f}")

## 1c. Кто и как работает (слайд 5) и зачем (слайд 6)

In [ ]:
order_t = ["none", "casual", "part", "full"]; nm_t = {"none": "не работает", "casual": "приработки", "part": "неполный день", "full": "полный день"}
pal = {"none": "#BBBBBB", "casual": "#9ECAE9", "part": "#4C78A8", "full": "#1F3A5F"}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for a, w in zip(ax[:2], ["3", "4"]):
    vc = d[f"wt{w}"].value_counts(normalize=True).reindex(order_t) * 100; n_ = int(d[f"wt{w}"].notna().sum())
    a.bar([nm_t[k] for k in order_t], vc.values, color=[pal[k] for k in order_t])
    for i, v in enumerate(vc.values): a.text(i, v, f"{v:.0f}%", ha="center", va="bottom")
    a.set_title(f"Тип занятости, волна {w} (n={n_})"); a.set_ylabel("% студентов"); a.tick_params(axis="x", rotation=18)
g = d[d.emp4 == 1]
sns.violinplot(data=g.assign(t=g.wt4.map(nm_t)), x="t", y="work_hours_4w", order=[nm_t[k] for k in order_t[1:]], palette=[pal[k] for k in order_t[1:]], cut=0, inner="quartile", ax=ax[2])
ax[2].set_title("Часы за прошлую неделю (волна 4)"); ax[2].set_xlabel(""); ax[2].set_ylabel("часов")
save("D01_who_works")
print(g.groupby("wt4").work_hours_4w.describe().round(1).rename(index=nm_t)[["count", "mean", "50%", "min", "max"]])
print(f"\nРаботают 21+ ч: {(g.work_hours_4w >= 21).mean()*100:.0f}% работающих; ≥35 ч: {(g.work_hours_4w >= 35).mean()*100:.0f}%")

In [ ]:
# Поток занятости 3 → 4 волна (аллювиальная диаграмма)
from matplotlib.patches import Rectangle
def alluvial(ax, left, right, order, colors, names, title):
    ct = pd.crosstab(left, right).reindex(index=order, columns=order, fill_value=0); N = ct.values.sum(); gap = .035 * N
    def pos(sizes):
        y, out = 0, {}
        for k, s in zip(order, sizes): out[k] = (y, y + s); y += s + gap
        return out
    lp, rp = pos(ct.sum(axis=1).values), pos(ct.sum(axis=0).values)
    xs = np.linspace(0, 1, 60); s_ = 3 * xs ** 2 - 2 * xs ** 3; x0, x1 = .10, .90
    lo, ro = {k: lp[k][0] for k in order}, {k: rp[k][0] for k in order}
    for a in order:
        for b in order:
            v = ct.loc[a, b]
            if v == 0: continue
            top = lo[a] + (ro[b] - lo[a]) * s_
            ax.fill_between(x0 + (x1 - x0) * xs, top, top + v, color=colors[a], alpha=.5, lw=0)
            lo[a] += v; ro[b] += v
    for k in order:
        ax.add_patch(Rectangle((x0 - .03, lp[k][0]), .03, lp[k][1] - lp[k][0], color=colors[k]))
        ax.add_patch(Rectangle((x1, rp[k][0]), .03, rp[k][1] - rp[k][0], color=colors[k]))
        ax.text(x0 - .045, (lp[k][0] + lp[k][1]) / 2, f"{names[k]}\n{int(ct.loc[k].sum())}", ha="right", va="center", fontsize=9)
        ax.text(x1 + .045, (rp[k][0] + rp[k][1]) / 2, f"{names[k]}\n{int(ct[k].sum())}", ha="left", va="center", fontsize=9)
    ax.set_xlim(-.18, 1.18); ax.set_ylim(max(lp[order[-1]][1], rp[order[-1]][1]) + gap, -gap); ax.axis("off")
    ax.text(x0 + .015, -gap * 1.2, "Волна 3", ha="center", fontweight="bold"); ax.text(x1 + .015, -gap * 1.2, "Волна 4", ha="center", fontweight="bold"); ax.set_title(title, pad=14)
fig, ax = plt.subplots(figsize=(9, 5.2))
alluvial(ax, P.wt3, P.wt4, order_t, pal, nm_t, f"Как менялась занятость у тех же студентов (n={len(P)})")
save("D01b_alluvial")
print(pd.crosstab(P.wt3.map(nm_t), P.wt4.map(nm_t), margins=True))

In [ ]:
# Мотивы, содержание и сфера работы (волна 4, только работающие)
x = d[d.emp4 == 1].copy()
mot_n = {1: "Оплачивать обучение", 2: "Содержать себя", 3: "Дополнительный доход", 4: "Получить опыт", 5: "Интерес к сфере"}
x["mot_n"] = x.work4_4w.map(mot_n); x["match_n"] = x.Work_major_4w.map({1: "совсем не соотв.", 2: "скорее не соотв.", 3: "скорее соотв.", 4: "полностью соотв."})
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.2]})
vc = x.mot_n.value_counts().reindex(list(mot_n.values())); ax[0].barh(vc.index[::-1], vc.values[::-1], color="#F58518")
for i, v in enumerate(vc.values[::-1]): ax[0].text(v, i, f" {int(v)}", va="center")
ax[0].set_title(f"Почему начали работать (n={int(vc.sum())})")
ct = pd.crosstab(x.wt4.map(nm_t), x.mot_n, normalize="index").reindex(index=[nm_t[k] for k in order_t[1:]], columns=list(mot_n.values())) * 100
ct.plot(kind="barh", stacked=True, ax=ax[1], colormap="Set2", legend=False); ax[1].set_title("Мотивы по типу занятости, %"); ax[1].set_xlabel("")
ax[1].legend(bbox_to_anchor=(0, -0.12), loc="upper left", ncol=2, fontsize=8)
vc2 = x.match_n.value_counts().reindex(["совсем не соотв.", "скорее не соотв.", "скорее соотв.", "полностью соотв."])
ax[2].bar(vc2.index, vc2.values, color=["#E45756", "#F2A29F", "#9ECAE9", "#4C78A8"]); ax[2].set_title(f"Соответствие работы специальности (n={int(vc2.sum())})")
for i, v in enumerate(vc2.values): ax[2].text(i, v, int(v), ha="center", va="bottom")
ax[2].tick_params(axis="x", rotation=15)
save("D01c_motives_match")
print("Мотив × соответствие специальности (% по строке):")
print((pd.crosstab(x.mot_n, x.match_n, normalize="index") * 100).round(0).reindex(list(mot_n.values())))
sp = x.copy(); sp["sphere"] = sp.work2_4w.map(VL["work2_4w"]); sp["high"] = (sp.Work_major_4w >= 3).astype(float).where(sp.Work_major_4w.notna())
g = sp.groupby("sphere").agg(n=("sphere", "size"), по_специальности=("high", "mean"), часы=("work_hours_4w", "mean")).sort_values("n", ascending=False)
print("\nСфера работы (n>=10):"); print(g[g.n >= 10].round(2).to_string())

## 2. ЦЕНА: чем жертвуют (H1)
### 2.1 Бюджет времени: что уступает место работе
План студента на 3-й курс (волна 3), проценты по шести видам деятельности. Разница работающих и неработающих с контролем на пол, финансирование, направление и материальное положение. Это **план**, а не фактическое время.

In [ ]:
PL = {"plan1": "Учёба в вузе", "plan2": "Клубы и организации", "plan3": "Оплачиваемая работа", "plan4": "Волонтёрство", "plan5": "Курсы вне вуза", "plan6": "Остальное (свободное время)"}
rows = []
for k, nm in PL.items():
    r = lpm(k, "emp3", d, "emp3", scale=1)
    r.update(вид=nm, не_работают=d.loc[d.emp3 == 0, k].mean(), работают=d.loc[d.emp3 == 1, k].mean()); rows.append(r)
tb = pd.DataFrame(rows).set_index("вид")
print(tb[["не_работают", "работают", "diff", "lo", "hi", "p", "n"]].round(2).to_string())
RES2.append(tb.reset_index().assign(блок="2.1 бюджет времени (п.п.)")[["блок", "вид", "diff", "lo", "hi", "p", "n"]])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
order_ = ["none", "casual", "part", "full"]; nm_ = {"none": "не работает", "casual": "приработки", "part": "неполный день", "full": "полный день"}
g = d[d.wt3.notna()].groupby("wt3")[list(PL)].mean().reindex(order_); g.index = [nm_[i] for i in g.index]; g.columns = list(PL.values())
g.plot(kind="barh", stacked=True, ax=ax[0], colormap="tab20c"); ax[0].set_xlabel("% планируемого времени"); ax[0].set_ylabel(""); ax[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=8)
ax[0].set_title("Планируемый бюджет времени (волна 3)")
t_ = tb.drop(index="Оплачиваемая работа")
cols_ = ["#E45756" if (r.p < .05 and r["diff"] < 0) else "#4C78A8" if r.p < .05 else "#BBBBBB" for _, r in t_.iterrows()]
ax[1].barh(t_.index, t_["diff"], xerr=[t_["diff"] - t_.lo, t_.hi - t_["diff"]], color=cols_, capsize=3)
ax[1].axvline(0, color="k", lw=.8); ax[1].set_xlabel("Разница работающих и неработающих, п.п. (95% ДИ)")
ax[1].set_title("Чем платят за работу\n(красный = значимое сокращение)")
save("D01_time_budget")

In [ ]:
# Зависит ли «цена» от мотива? (план времени работающих по мотиву, 3 волна)
x = d[(d.emp3 == 1) & d.motive3.isin(["money", "exp"])]
t = x.groupby("motive3")[list(PL)].mean().rename(columns=PL).round(1).rename(index={"money": "ради денег", "exp": "ради опыта/интереса"})
t["n"] = x.motive3.value_counts().rename(index={"money": "ради денег", "exp": "ради опыта/интереса"})
print(t.to_string())
for k, nm in PL.items():
    a = x.loc[x.motive3 == "money", k].dropna(); b = x.loc[x.motive3 == "exp", k].dropna()
    print(f"  {nm}: разница денег−опыт = {a.mean()-b.mean():+.1f} п.п., p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.2f}")

### 2.2 Сон, здоровье, учёба: сводный график «цена и выгода по исходам»
Разница работающих и неработающих в п.п. (для сна в минутах), с контролями, отдельно по двум волнам. Стрелка в подписи показывает, в какую сторону лучше. Совпадение двух волн — аргумент в пользу устойчивости.

In [ ]:
OUT = [("sleep_avg", "Сон, минут в сутки (↑ лучше)", 60), ("short_sleep", "Сон <7 ч (↑ хуже)", 100), ("sport", "Занимается спортом (↑ лучше)", 100),
       ("smoke", "Курит (↑ хуже)", 100), ("alco", "Пьёт алкоголь (↑ хуже)", 100), ("drop", "Думал об отчислении (↑ хуже)", 100),
       ("mark_hi", "Оценки только 4–5 (↑ лучше)", 100), ("sat_hi", "Удовлетворён обучением (↑ лучше)", 100), ("again", "Поступил бы снова (↑ лучше)", 100)]
rows = []
for base, nm, sc in OUT:
    for w in ["3", "4"]:
        r = lpm(f"{base}_{w}w", f"emp{w}", d, f"emp{w}", scale=sc); r.update(исход=nm, волна=w); rows.append(r)
cost = pd.DataFrame(rows)
RES2.append(cost.assign(блок="2.2 исходы работающие−неработающие", вид=cost.исход + " | волна " + cost.волна)[["блок", "вид", "diff", "lo", "hi", "p", "n"]])
fig, ax = plt.subplots(figsize=(9, 5.5)); ys = {nm: i for i, (_, nm, _) in enumerate(OUT[::-1])}
for w, col, off in [("3", "#4C78A8", .14), ("4", "#F58518", -.14)]:
    s = cost[cost.волна == w]
    ax.errorbar(s["diff"], [ys[o] + off for o in s.исход], xerr=[s["diff"] - s.lo, s.hi - s["diff"]], fmt="o", color=col, capsize=3, label=f"волна {w}")
ax.axvline(0, color="k", lw=.8); ax.set_yticks(list(ys.values())); ax.set_yticklabels(list(ys.keys()))
ax.set_xlabel("Работают минус не работают: п.п. (сон: минуты), 95% ДИ"); ax.legend(); ax.set_title("Цена и выгода работы по исходам (с контролями)")
save("D02_cost_benefit_outcomes")
cost.assign(знач=cost.p.apply(stars)).pivot_table(index="исход", columns="волна", values=["diff", "p"], aggfunc="first").round(2)

## 3. ВЫГОДА и от чего она зависит: содержание, интенсивность, мотив (H2–H4)
Работающие 4-й волны (самооценки спрашивали только их). Три модели на каждый исход:
1. **аддитивная:** соответствие специальности + 21+ ч + мотив + контроли (мотив оценивается *при* учёте содержания и часов);
2. **с взаимодействием** содержание × интенсивность (тест: отличается ли «цена часов» для работы по и не по специальности);
3. **профили 2×2** — четыре типа работающих по двум осям.

In [ ]:
YS = {"self_help": "Работа помогает усваивать материал", "self_hurt": "Из-за работы страдает успеваемость", "self_hard": "Очень трудно совмещать",
      "drop_4w": "Думал об отчислении", "mark_hi_4w": "Оценки только 4–5", "sat_hi_4w": "Удовлетворён обучением", "short_sleep_4w": "Сон <7 ч", "again_4w": "Поступил бы снова"}
rows, inter = [], []
for y, nm in YS.items():
    add = f"{y} ~ low_match + h21 + exp_mot + {CTRL}"
    try:
        m = smf.logit(add, w4).fit(disp=0, cov_type="HC1")
        ci = m.conf_int()
        for t in ["low_match", "h21", "exp_mot"]:
            rows.append(dict(исход=nm, терм=t, OR=np.exp(m.params[t]), lo=np.exp(ci.loc[t, 0]), hi=np.exp(ci.loc[t, 1]), p=m.pvalues[t], n=int(m.nobs)))
        m0 = smf.logit(add, w4).fit(disp=0); m1 = smf.logit(f"{y} ~ low_match*h21 + exp_mot + {CTRL}", w4).fit(disp=0)
        inter.append(dict(исход=nm, p_взаимод=stats.chi2.sf(2 * (m1.llf - m0.llf), 1)))
    except Exception as e:
        print("[!]", nm, str(e)[:70])
A = pd.DataFrame(rows)
lab = {"low_match": "Работа НЕ по специальности (vs по)", "h21": "21+ ч (vs ≤20)", "exp_mot": "Мотив «опыт/интерес» (vs деньги)"}
A["эффект"] = A.OR.round(2).astype(str) + A.p.apply(stars)
print("Отношения шансов, все три фактора одновременно (с контролями); * p<.05, ** p<.01, *** p<.001")
print(A.pivot_table(index="исход", columns="терм", values="эффект", aggfunc="first").rename(columns=lab).loc[list(YS.values())].to_string())
print("\nn:", int(A.n.min()), "–", int(A.n.max()))
print("\nТест взаимодействия (содержание × интенсивность), p-value:")
print(pd.DataFrame(inter).round(3).to_string(index=False))
RES2.append(A.assign(блок="3 аддитивная модель (OR)", вид=A.исход + " | " + A.терм, diff=A.OR)[["блок", "вид", "diff", "lo", "hi", "p", "n"]])

In [ ]:
# Профили 2×2: доли и 95% ДИ Уилсона
from statsmodels.stats.proportion import proportion_confint
order_p = [prof[(0, 0)], prof[(0, 1)], prof[(1, 0)], prof[(1, 1)]]
S3 = {"self_help": ("Помогает усваивать материал", "#54A24B"), "self_hurt": ("Страдает успеваемость", "#E45756"), "self_hard": ("Очень трудно совмещать", "#F58518")}
fig, ax = plt.subplots(figsize=(11, 4.8)); wdt = .26
for j, (y, (nm, col)) in enumerate(S3.items()):
    ps, los, his = [], [], []
    for pr in order_p:
        s = w4.loc[w4.profile == pr, y].dropna(); k, n = int(s.sum()), len(s)
        lo, hi = proportion_confint(k, n, method="wilson"); ps.append(k / n); los.append(k / n - lo); his.append(hi - k / n)
    ax.bar(np.arange(4) + (j - 1) * wdt, np.array(ps) * 100, wdt, yerr=[np.array(los) * 100, np.array(his) * 100], color=col, capsize=3, label=nm)
nn = [int((w4.profile == pr).sum()) for pr in order_p]
ax.set_xticks(range(4)); ax.set_xticklabels([p_.replace(" (", "\n(") + f"\nn={n}" for p_, n in zip(order_p, nn)], fontsize=9)
ax.set_ylabel("% согласных (95% ДИ)"); ax.set_ylim(0, 100); ax.legend(); ax.set_title("Польза и цена совмещения по профилям работающих (волна 4)")
save("D03_profiles_benefit_cost")

> **Как читать.** Верхняя строка таблицы с OR: `low_match` показывает, как меняется шанс согласиться с утверждением, если работа не по специальности. Для «помогает усваивать» ждём OR << 1 (H2). `h21` для «страдает успеваемость» и «трудно совмещать» ждём OR > 1 (H3). `exp_mot` смотрим при уже учтённых содержании и часах (H4): если OR близок к 1, мотив работал как прокси содержания. Тест взаимодействия: значимый p означает, что цена часов различается для работы по и не по специальности.

## 4. Механизм: интенсивность → ощущение конфликта → мысли об отчислении (H5)
Простой анализ посредничества на работающих 4-й волны: прямой эффект и эффект через «трудно совмещать» (произведение коэффициентов a·b, бутстрэп 95% ДИ). Данные срезовые, поэтому это **описание согласованности с механизмом**, а не доказательство причинности.

In [ ]:
def boot_med(data, expo, B=1000, seed=0):
    X = data[[expo, "self_hard", "drop_4w", "sex_c", "funding_c", "major_c", "ses_c"]].dropna()
    ya, Xa = patsy.dmatrices(f"self_hard ~ {expo} + {CTRL}", X, return_type="dataframe")
    yb, Xb = patsy.dmatrices(f"drop_4w ~ self_hard + {expo} + {CTRL}", X, return_type="dataframe")
    yc, Xc = patsy.dmatrices(f"drop_4w ~ {expo} + {CTRL}", X, return_type="dataframe")
    ia, ib, ib2, ic = Xa.columns.get_loc(expo), Xb.columns.get_loc("self_hard"), Xb.columns.get_loc(expo), Xc.columns.get_loc(expo)
    Xa_, ya_, Xb_, yb_, Xc_, yc_ = [np.asarray(z, float) for z in (Xa, ya, Xb, yb, Xc, yc)]
    def est(ix):
        a = np.linalg.lstsq(Xa_[ix], ya_[ix], rcond=None)[0][ia, 0]
        bb = np.linalg.lstsq(Xb_[ix], yb_[ix], rcond=None)[0]
        c = np.linalg.lstsq(Xc_[ix], yc_[ix], rcond=None)[0][ic, 0]
        return a, bb[ib, 0], bb[ib2, 0], c
    n = len(X); a, b, cp, c = est(np.arange(n)); rng = np.random.default_rng(seed)
    bs = np.array([est(rng.integers(0, n, n)) for _ in range(B)])
    ind = bs[:, 0] * bs[:, 1]; lo, hi = np.percentile(ind, [2.5, 97.5])
    return dict(экспозиция=expo, n=n, a_exposure_to_conflict_pp=a * 100, b_conflict_to_dropout_pp=b * 100, total_pp=c * 100, direct_pp=cp * 100,
                indirect_pp=a * b * 100, ind_lo=lo * 100, ind_hi=hi * 100, доля_через_конфликт=(a * b / c) if abs(c) > 1e-9 else np.nan)
med = pd.DataFrame([boot_med(w4, "h21"), boot_med(w4, "low_match")]).round(3)
med["экспозиция"] = med["экспозиция"].map({"h21": "21+ ч (vs ≤20)", "low_match": "не по специальности (vs по)"})
print(med.T.to_string(header=False))
RES2.append(med.assign(блок="4 посредничество (п.п.)", вид=med["экспозиция"], diff=med.indirect_pp, lo=med.ind_lo, hi=med.ind_hi, p=np.nan)[["блок", "вид", "diff", "lo", "hi", "p", "n"]])
# Причины мыслей об отчислении у думавших: доли
print()
for w in ["3w", "4w"]:
    wk = "emp3" if w == "3w" else "emp4"
    t = d[d[f"r_work_{w}"].notna()].groupby(wk)[[f"r_work_{w}", f"r_econ_{w}"]].mean().mul(100).round(0)
    t.columns = ["трудности совмещения, %", "экономические трудности, %"]; print(f"Причины у думавших об отчислении, {w}:"); print(t.rename(index={0.0: "не работает", 1.0: "работает"}))

> **Как читать.** `indirect_pp` — на сколько п.п. выше доля думавших об отчислении у студентов с 21+ ч *через* ощущение конфликта; `direct_pp` — остаток. Если ДИ непрямого эффекта не включает 0, механизм согласуется с данными. `доля_через_конфликт` — сколько процентов общей связи он «объясняет».

## 5. ОТБОР ДО РАБОТЫ: кто начинает работать (H6)
Берём тех, кто **не работал в 3-й волне и ответил в 4-й** (n≈342; 94 начали). Смотрим, отличались ли будущие работающие *до* начала работы: по намерениям, плану времени, успеваемости, сну, привычкам, происхождению.

In [ ]:
Bn = d[(d.emp3 == 0) & d.emp4.notna()].copy()
Bn["start"] = (Bn.emp4 == 1).astype(float)
Bn["plan_work10"], Bn["plan_study10"] = Bn.plan3 / 10, Bn.plan1 / 10
print("Выборка:", len(Bn), "| начали работать:", int(Bn.start.sum()), f"({Bn.start.mean()*100:.0f}%)\n")

# Описательно
print("Доля начавших работать по намерению искать работу (3в):")
print(Bn.groupby(Bn.intent.map({1.0: "планировал искать", 0.0: "не планировал"}))["start"].agg(["mean", "count"]).round(2))
Bn["plan_work_cat"] = pd.cut(Bn.plan3, [-1, 0, 10, 100], labels=["0%", "1–10%", "11%+"])
print("\nПо планируемой доле времени на оплачиваемую работу (3в):")
print(Bn.groupby("plan_work_cat")["start"].agg(["mean", "count"]).round(2))
print("\nПо стажировке (3в):"); print(Bn.groupby(Bn.intern_3w.map({1.0: "была стажировка", 0.0: "не было"}))["start"].agg(["mean", "count"]).round(2))

# Баланс до начала: чем отличались будущие работающие (3 волна)
bal = {"Оценки 4–5": "mark_hi_3w", "Удовлетворён": "sat_hi_3w", "Думал об отчислении": "drop_3w", "Сон, ч": "sleep_avg_3w", "Social jetlag": "sjl_3w",
       "Спорт": "sport_3w", "Курит": "smoke_3w", "Пьёт": "alco_3w", "План: учёба %": "plan1", "План: работа %": "plan3", "План: свободное %": "plan6", "Намерен искать работу": "intent"}
rows = []
for nm, v in bal.items():
    a = Bn.loc[Bn.start == 1, v].dropna(); b = Bn.loc[Bn.start == 0, v].dropna()
    rows.append({"показатель (3в)": nm, "начали": a.mean(), "не начали": b.mean(), "разница": a.mean() - b.mean(), "p": stats.ttest_ind(a, b, equal_var=False).pvalue})
print("\nЧем отличались ДО начала (средние/доли):"); print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# Баланс до начала работы: стандартизованные разницы (Love plot). |SMD| > 0.2 = заметное различие
dm = pd.get_dummies(Bn[["sex_c", "funding_c", "major_c", "ses_c"]]).astype(float)
dm = dm[[c for c in dm.columns if not c.endswith("_na")]]
vv = Bn[["mark_hi_3w", "sat_hi_3w", "drop_3w", "sleep_avg_3w", "sjl_3w", "sport_3w", "smoke_3w", "alco_3w", "plan1", "plan3", "plan6", "intent", "intern_3w"]].join(dm)
nm_ = {"mark_hi_3w": "Оценки 4–5", "sat_hi_3w": "Удовлетворён", "drop_3w": "Думал об отчислении", "sleep_avg_3w": "Сон, ч", "sjl_3w": "Social jetlag", "sport_3w": "Спорт",
       "smoke_3w": "Курит", "alco_3w": "Пьёт", "plan1": "План: учёба", "plan3": "План: работа", "plan6": "План: свободное", "intent": "Намерен искать работу", "intern_3w": "Была стажировка",
       "sex_c_f": "Женский пол", "sex_c_m": "Мужской пол", "funding_c_paid": "Платное обучение", "funding_c_budget": "Бюджет", "funding_c_target": "Целевое",
       "major_c_eng": "Направление: инженерное", "major_c_math": "Направление: мат/ест.", "major_c_soc": "Направление: социальное", "major_c_hum": "Направление: гуманитарное", "major_c_oth": "Направление: другое",
       "ses_c_low": "Материально: низкое", "ses_c_mid": "Материально: среднее", "ses_c_high": "Материально: высокое", "ses_c_vhigh": "Материально: очень высокое"}
smd = []
for c in vv.columns:
    a = vv.loc[Bn.start == 1, c].dropna(); b = vv.loc[Bn.start == 0, c].dropna()
    sd = np.sqrt((a.var() + b.var()) / 2)
    if sd > 0: smd.append((nm_.get(c, c), (a.mean() - b.mean()) / sd, stats.ttest_ind(a, b, equal_var=False).pvalue))
smd = pd.DataFrame(smd, columns=["показатель", "SMD", "p"]).sort_values("SMD", key=abs)
fig, ax = plt.subplots(figsize=(8, 8)); ax.scatter(smd.SMD, range(len(smd)), c=["#E45756" if p < .05 else "#4C78A8" for p in smd.p], s=40, zorder=3)
ax.hlines(range(len(smd)), 0, smd.SMD, color="#CCCCCC", zorder=1)
for v in (-.2, .2): ax.axvline(v, color="k", ls="--", lw=.8)
ax.axvline(0, color="k", lw=.8); ax.set_yticks(range(len(smd))); ax.set_yticklabels(smd.показатель, fontsize=8)
ax.set_xlabel("Стандартизованная разница: начали работать − не начали (красный: p<.05)"); ax.set_title(f"Чем будущие работающие отличались до начала (n={len(Bn)}, начали {int(Bn.start.sum())})")
save("D04b_love_plot")
print("Показателей с |SMD|>0.2:", int((smd.SMD.abs() > .2).sum()), "из", len(smd))

In [ ]:
# Многомерная модель: что предсказывает старт (OR, 95% ДИ)
f = f"start ~ intent + plan_work10 + plan_study10 + intern_3w + mark_hi_3w + sat_hi_3w + drop_3w + sleep_avg_3w + smoke_3w + {CTRL}"
m = smf.logit(f, Bn).fit(disp=0, cov_type="HC1")
ci = m.conf_int(); K = ["intent", "plan_work10", "plan_study10", "intern_3w", "mark_hi_3w", "sat_hi_3w", "drop_3w", "sleep_avg_3w", "smoke_3w"]
nmK = {"intent": "Намерен искать работу", "plan_work10": "План: доля работы (+10 п.п.)", "plan_study10": "План: доля учёбы (+10 п.п.)", "intern_3w": "Была стажировка",
       "mark_hi_3w": "Оценки 4–5", "sat_hi_3w": "Удовлетворён обучением", "drop_3w": "Думал об отчислении", "sleep_avg_3w": "Сон (+1 ч)", "smoke_3w": "Курит"}
R = pd.DataFrame({"OR": np.exp(m.params[K]), "lo": np.exp(ci.loc[K, 0]), "hi": np.exp(ci.loc[K, 1]), "p": m.pvalues[K]}).rename(index=nmK)
print("n =", int(m.nobs)); print(R.round(3).to_string())
RES2.append(R.reset_index().rename(columns={"index": "вид", "OR": "diff"}).assign(блок="5 предикторы старта (OR)", n=int(m.nobs))[["блок", "вид", "diff", "lo", "hi", "p", "n"]])
fig, ax = plt.subplots(figsize=(8, 4.5)); r = R.iloc[::-1]
cols_ = ["#4C78A8" if p < .05 else "#BBBBBB" for p in r.p]
ax.errorbar(r.OR, range(len(r)), xerr=[r.OR - r.lo, r.hi - r.OR], fmt="none", ecolor="#888", capsize=3)
ax.scatter(r.OR, range(len(r)), c=cols_, zorder=3, s=45); ax.axvline(1, color="k", lw=.8); ax.set_xscale("log")
ax.set_yticks(range(len(r))); ax.set_yticklabels(r.index); ax.set_xlabel("Отношение шансов начать работать (95% ДИ)"); ax.set_title("Кто начинает работать (по данным 3-й волны)")
save("D04_road_to_work")

In [ ]:
# Насколько хорошо ранние признаки предсказывают старт? Кросс-валидированная AUC
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
    yy, XX = patsy.dmatrices("start ~ intent + plan_work10 + plan_study10 + intern_3w + mark_hi_3w + sat_hi_3w + drop_3w + sleep_avg_3w + smoke_3w + " + CTRL, Bn, return_type="dataframe")
    yy = yy.iloc[:, 0].values; XX = XX.drop(columns="Intercept")
    dem = [c for c in XX.columns if c.startswith("C(")]
    sets = {"только намерение": ["intent"], "+ план времени": ["intent", "plan_work10", "plan_study10"],
            "+ успеваемость, сон, привычки": ["intent", "plan_work10", "plan_study10", "intern_3w", "mark_hi_3w", "sat_hi_3w", "drop_3w", "sleep_avg_3w", "smoke_3w"],
            "+ демография": list(XX.columns), "только демография": dem}
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0)
    for nm, cols_ in sets.items():
        sc = cross_val_score(LogisticRegression(max_iter=2000, C=1.0), (XX[cols_] - XX[cols_].mean()) / XX[cols_].std().replace(0, 1), yy, cv=cv, scoring="roc_auc")
        print(f"{nm:34s} AUC = {sc.mean():.3f} ± {sc.std():.3f}   (n = {len(yy)})")
    print("\nAUC 0.5 = угадывание, 0.7 = приемлемо, 0.8+ = хорошо.")
except Exception as e:
    print("sklearn недоступен:", e)

In [ ]:
# Осыпание и блок 5: пересчёт модели старта с весами обратной вероятности (IPW)
G = d[d.emp3 == 0].copy(); G["resp4"] = G.emp4.notna().astype(float)
for v, nm in [("mark_3w", "mk"), ("satisfaction_3w", "st"), ("dropout1_3w", "dr")]: G[nm + "_c"] = G[v].map(lambda z: "na" if pd.isna(z) else str(int(z)))
G["int_c"] = G.intent.map(lambda z: "na" if pd.isna(z) else str(int(z)))
G["pw_c"] = pd.cut(G.plan3, [-1, 0, 10, 100], labels=["0", "1_10", "11p"]).astype(object).fillna("na")
att = smf.logit(f"resp4 ~ C(int_c) + C(pw_c) + C(mk_c) + C(st_c) + C(dr_c) + {CTRL}", G).fit(disp=0)
G["pr"] = att.predict(G); G["w"] = G.resp4.mean() / G.pr
lo_, hi_ = G.loc[G.resp4 == 1, "w"].quantile([.01, .99]); G["w"] = G.w.clip(lo_, hi_)
Bw = Bn.join(G["w"], how="left")
print(f"Неработавшие в 3в: {len(G)}, ответили в 4в: {int(G.resp4.sum())} ({G.resp4.mean()*100:.0f}%) | веса: mean={Bw.w.mean():.2f}, min={Bw.w.min():.2f}, max={Bw.w.max():.2f}")
f = f"start ~ intent + plan_work10 + plan_study10 + intern_3w + mark_hi_3w + sat_hi_3w + drop_3w + sleep_avg_3w + smoke_3w + {CTRL}"
m_w = smf.glm(f, Bw.dropna(subset=["w"]), family=sm.families.Binomial(), freq_weights=Bw.dropna(subset=["w"]).w).fit(cov_type="HC1")
m_u = smf.logit(f, Bn).fit(disp=0, cov_type="HC1")
cmpr = pd.DataFrame({"OR без весов": np.exp(m_u.params[K]), "p": m_u.pvalues[K], "OR IPW": np.exp(m_w.params[K]), "p IPW": m_w.pvalues[K]}).rename(index=nmK).round(3)
print(cmpr.to_string())

In [ ]:
# Кто бросает работу? (работали в 3 волне, ответили в 4-й). Малая выборка: компактный набор контролей
Qd = d[(d.emp3 == 1) & d.emp4.notna()].copy(); Qd["quit"] = (Qd.emp4 == 0).astype(float); Qd["h10"] = Qd.work_hours / 10
print("Работали в 3в и ответили в 4в:", len(Qd), "| бросили:", int(Qd.quit.sum()))
try:
    f = "quit ~ h10 + C(motive3, Treatment('money')) + sat_hi_3w + drop_3w + mark_hi_3w + intern_3w + C(sex_c)"
    mq = smf.logit(f, Qd).fit(disp=0, cov_type="HC1"); ci = mq.conf_int()
    kq = [t for t in mq.params.index if any(s in t for s in ["h10", "motive3", "sat_hi", "drop_3w", "mark_hi", "intern"])]
    print(pd.DataFrame({"OR": np.exp(mq.params[kq]), "lo": np.exp(ci.loc[kq, 0]), "hi": np.exp(ci.loc[kq, 1]), "p": mq.pvalues[kq]}).round(3).to_string()); print("n =", int(mq.nobs))
except Exception as e:
    print("Модель не сошлась:", str(e)[:80])
# Проверка «курение у начавших работать» (аномальный результат ноутбука 2)
Pn = d[d.panel & (d.smoke_3w == 0) & d.smoke_4w.notna()]
print("\nНе курили в 3в -> курят в 4в, по переходам занятости:")
t = pd.crosstab(Pn.trans, Pn.smoke_4w.map({0.0: "не курит", 1.0: "курит"})); t["% начали курить"] = (t["курит"] / t.sum(axis=1) * 100).round(1); print(t)
a = Pn[Pn.trans == "start"].smoke_4w; b = Pn[Pn.trans == "stay_no"].smoke_4w
print("Fisher (start vs stay_no): p =", round(stats.fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])[1], 3))

> **Как читать.** Если `intent` и `plan_work10` значимы, а оценки, сон и демография нет, получается красивый вывод: **выбор работы определяется намерениями и планами, а не успеваемостью и происхождением**. AUC показывает, насколько этот выбор предсказуем. В блоке про курение смотрите абсолютные числа: если «начали курить» всего несколько человек, результат ненадёжен и выносить его на слайд не стоит.

## 6. Профили совмещающих студентов
Четыре типа работающих по двум осям (содержание × интенсивность), каждый описан по мотивам, цене и выгоде. Это описательная типология; ниже проверяем, что она не противоречит автоматической кластеризации.

In [ ]:
pt = w4[w4.profile.notna()].groupby("profile").agg(
    n=("profile", "size"),
    часы=("work_hours_4w", "mean"),
    полный_день=("wt4", lambda s: (s == "full").mean()),
    мотив_опыт=("exp_mot", "mean"),
    мотив_содержать_себя=("work4_4w", lambda s: (s == 2).mean()),
    стажировка=("intern1_4w", lambda s: (s == 1).mean()),
    помогает=("self_help", "mean"), страдает=("self_hurt", "mean"), трудно=("self_hard", "mean"),
    оценки_4_5=("mark_hi_4w", "mean"), отчисление=("drop_4w", "mean"), удовлетворён=("sat_hi_4w", "mean"), сон_ч=("sleep_avg_4w", "mean"),
).reindex(order_p)
print(pt.round(2).T.to_string())
ph = pt.drop(columns=["n", "часы", "сон_ч"]).astype(float)
fig, ax = plt.subplots(figsize=(8.5, 5)); sns.heatmap(ph.T, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax, annot_kws={"fontsize": 9})
ax.set_xticklabels([p_.replace(" (", "\n(") for p_ in order_p], rotation=0, fontsize=8); ax.set_title("Профили совмещающих студентов: доли"); save("D05_profiles_heatmap")
RES2.append(pt.reset_index().melt(id_vars="profile").assign(блок="6 профили", вид=lambda x: x.profile + " | " + x.variable, diff=lambda x: x.value, lo=np.nan, hi=np.nan, p=np.nan, n=np.nan)[["блок", "вид", "diff", "lo", "hi", "p", "n"]])

In [ ]:
# Робастность: автоматическая кластеризация (k-means) vs профили 2×2
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, adjusted_rand_score
    Z = w4[["work_hours_4w", "Work_major_4w", "exp_mot", "profile"]].dropna()
    F = Z[["work_hours_4w", "Work_major_4w", "exp_mot"]]; F = (F - F.mean()) / F.std()
    for k in [2, 3, 4, 5]:
        km = KMeans(k, n_init=20, random_state=0).fit(F)
        print(f"k={k}: silhouette = {silhouette_score(F, km.labels_):.3f}")
    km = KMeans(4, n_init=20, random_state=0).fit(F)
    print("\nСогласие k-means (k=4) с профилями 2×2, adjusted Rand =", round(adjusted_rand_score(Z.profile, km.labels_), 3), "(0 = случайно, 1 = полное совпадение)")
    print(pd.crosstab(Z.profile, km.labels_))
except Exception as e:
    print("sklearn недоступен:", e)

## 7. Повторная проверка H2 на стажировках (3 волна)
Содержательный эффект (связь с направлением → ощущение пользы) должен воспроизводиться и на другом виде совмещения. В 3-й волне студенты оценивали стажировку по тем же четырём утверждениям.

In [ ]:
I = d[d.intern1 == 1].copy()
I["related"] = np.where(I.intern5_1.isna(), np.nan, (I.intern5_1 >= 3).astype(float))
I["paid"]    = np.where(I.intern3.isna(), np.nan, (I.intern3 == 1).astype(float))
for k, nm in [(2, "help"), (3, "hurt"), (4, "hard")]: I[nm] = np.where(I[f"intern5_{k}"].isna(), np.nan, (I[f"intern5_{k}"] >= 3).astype(float))
print("Стажировки (3в): n =", len(I), "| по направлению:", int(I.related.sum()), "| оплачиваемые:", int(I.paid.sum()))
print("\nДоли согласных, по связи стажировки с направлением:")
print(I.groupby(I.related.map({1.0: "по направлению", 0.0: "не по направлению"}))[["help", "hurt", "hard"]].mean().mul(100).round(0).join(I.groupby(I.related.map({1.0: "по направлению", 0.0: "не по направлению"})).size().rename("n")))
rows = []
for y, nm in [("help", "Помогает усваивать"), ("hurt", "Страдает успеваемость"), ("hard", "Трудно совмещать")]:
    m = smf.logit(f"{y} ~ related + paid + {CTRL}", I).fit(disp=0, cov_type="HC1"); ci = m.conf_int()
    for t in ["related", "paid"]:
        rows.append(dict(исход=nm, терм=t, OR=np.exp(m.params[t]), lo=np.exp(ci.loc[t, 0]), hi=np.exp(ci.loc[t, 1]), p=m.pvalues[t], n=int(m.nobs)))
IR = pd.DataFrame(rows); print("\nOR (по направлению / оплачиваемая):"); print(IR.round(3).to_string(index=False))
RES2.append(IR.assign(блок="7 стажировки (OR)", вид=IR.исход + " | " + IR.терм, diff=IR.OR)[["блок", "вид", "diff", "lo", "hi", "p", "n"]])
# Сравнение: стажировка (3в) vs работа (4в), доли согласных (разные выборки и волны, только как ориентир)
print("\nОриентир: стажировка (3в) vs работа (4в), % согласных")
print(pd.DataFrame({"стажировка 3в": [I.help.mean(), I.hurt.mean(), I.hard.mean()], "работа 4в": [w4.self_help.mean(), w4.self_hurt.mean(), w4.self_hard.mean()]},
                   index=["помогает", "страдает", "трудно"]).mul(100).round(0))

## 7b. Устойчивость, множественные сравнения, E-value
1. Меняется ли главный результат, если по-другому задать порог часов (15/20/25) или строже определить «по специальности» (только «полностью соответствует»)?
2. Порядковые модели вместо бинарных (используется вся шкала 1–4).
3. Поправка Бенджамини–Хохберга на 24 проверки блока 3.
4. E-value: насколько сильным должен быть ненаблюдаемый фактор, чтобы свести эффект к нулю.

In [ ]:
rows = []
for thr in [15, 20, 25, 30]:
    w4["hthr"] = np.where(w4.work_hours_4w.notna(), (w4.work_hours_4w >= thr).astype(float), np.nan)
    for y in ["self_hurt", "self_hard", "drop_4w"]:
        m = smf.logit(f"{y} ~ low_match + hthr + exp_mot + {CTRL}", w4).fit(disp=0, cov_type="HC1"); ci = m.conf_int()
        rows.append(dict(порог_часов=f"≥{thr}", исход=YS[y], OR=np.exp(m.params["hthr"]), lo=np.exp(ci.loc["hthr", 0]), hi=np.exp(ci.loc["hthr", 1]), p=m.pvalues["hthr"], n=int(m.nobs), доля_выше_порога=w4.hthr.mean()))
T1 = pd.DataFrame(rows); print("OR «работает ≥ порога часов» при разных порогах (с учётом соответствия и мотива):")
print(T1.pivot_table(index="исход", columns="порог_часов", values="OR").round(2).to_string())
print(T1.pivot_table(index="исход", columns="порог_часов", values="p").round(3).to_string())
# строгое определение «по специальности»
w4["low_strict"] = np.where(w4.Work_major_4w.notna(), (w4.Work_major_4w <= 3).astype(float), np.nan)
print("\nСтрогое определение: «полностью соответствует» (n=%d) vs остальные (n=%d)" % ((w4.low_strict == 0).sum(), (w4.low_strict == 1).sum()))
for y in ["self_help", "self_hurt", "self_hard"]:
    m = smf.logit(f"{y} ~ low_strict + h21 + exp_mot + {CTRL}", w4).fit(disp=0, cov_type="HC1"); print(f"  {YS[y]}: OR(не полностью) = {np.exp(m.params['low_strict']):.2f}, p={m.pvalues['low_strict']:.3f}")

In [ ]:
# Порядковые логит-модели: вся шкала согласия 1–4 вместо бинарного «согласен»
for y, nm in [("self_help_ord", "Помогает усваивать"), ("self_hurt_ord", "Страдает успеваемость"), ("self_hard_ord", "Трудно совмещать")]:
    fit(f"{y} ~ low_match + h21 + exp_mot + {CTRL}", w4, "ordered", "ord_self", nm, ["low_match", "h21", "exp_mot"])
print(show("ord_self")[["outcome", "term", "n", "OR", "OR_lo", "OR_hi", "p"]].to_string(index=False))

# Поправка BH для семейства блока 3 и E-value
A["p_BH"] = multipletests(A.p, method="fdr_bh")[1]
print(f"\nСемейство блока 3: {len(A)} проверок. Значимо без поправки: {(A.p < .05).sum()}; после BH (q<.05): {(A.p_BH < .05).sum()}")
print(A[A.p < .10][["исход", "терм", "OR", "p", "p_BH"]].round(3).to_string(index=False))
def evalue(or_, lo, hi):
    rr = np.sqrt(or_) if or_ >= 1 else np.sqrt(1 / or_); ev = rr + np.sqrt(rr * (rr - 1))
    if (or_ >= 1 and lo <= 1) or (or_ < 1 and hi >= 1): return ev, 1.0
    cl = lo if or_ >= 1 else hi; rc = np.sqrt(cl) if or_ >= 1 else np.sqrt(1 / cl); return ev, rc + np.sqrt(rc * (rc - 1))
ev = A[A.p < .05].copy(); ev[["E-value", "E-value (ДИ)"]] = ev.apply(lambda r: pd.Series(evalue(r.OR, r.lo, r.hi)), axis=1)
print("\nE-value для значимых оценок блока 3 (чем больше, тем устойчивее к скрытым факторам):")
print(ev[["исход", "терм", "OR", "E-value", "E-value (ДИ)"]].round(2).to_string(index=False))

In [ ]:
# Автотезисы для слайдов: числа подставляются из расчётов
r = lambda df_, name: df_.loc[name]
tb_ = tb["diff"]; ap = A.set_index(["исход", "терм"])
print("СЛАЙД 7 (цена):")
print(f"  • Работающие планируют отдавать на учёбу на {abs(tb_['Учёба в вузе']):.0f} п.п. меньше, на свободное время на {abs(tb_['Остальное (свободное время)']):.0f} п.п. меньше; клубы, курсы и волонтёрство не отличаются.")
cs = cost.set_index(["исход", "волна"])
print(f"  • Сон короче на {abs(cs.loc[('Сон, минут в сутки (↑ лучше)', '3'), 'diff']):.0f} и {abs(cs.loc[('Сон, минут в сутки (↑ лучше)', '4'), 'diff']):.0f} мин (волны 3 и 4).")
print(f"  • Оценки и удовлетворённость в среднем не отличаются (p={cs.loc[('Оценки только 4–5 (↑ лучше)', '4'), 'p']:.2f} и p={cs.loc[('Удовлетворён обучением (↑ лучше)', '4'), 'p']:.2f}, волна 4).")
print("СЛАЙД 8 (выгода и от чего зависит):")
o = ap.loc[("Работа помогает усваивать материал", "low_match")]; print(f"  • Работа не по специальности: шансы считать работу полезной для учёбы ниже в {1/o.OR:.1f} раза (OR={o.OR:.2f}, 95% ДИ {o.lo:.2f}–{o.hi:.2f}).")
o = ap.loc[("Из-за работы страдает успеваемость", "h21")]; print(f"  • 21+ часов: шансы «страдает успеваемость» выше в {o.OR:.1f} раза (95% ДИ {o.lo:.2f}–{o.hi:.2f}).")
o = ap.loc[("Из-за работы страдает успеваемость", "exp_mot")]; print(f"  • Мотив «опыт/интерес» (при учёте содержания и часов): OR={o.OR:.2f} для «страдает успеваемость».")
print(f"  • Через ощущение конфликта проходит около {med.loc[0, 'доля_через_конфликт']*100:.0f}% связи 21+ часов с мыслями об отчислении.")
print("СЛАЙД 9 (отбор до работы):")
print(f"  • Начали работать {int(Bn.start.sum())} из {len(Bn)}; среди планировавших искать работу {Bn[Bn.intent==1].start.mean()*100:.0f}%, среди остальных {Bn[Bn.intent==0].start.mean()*100:.0f}%.")
print(f"  • Намерение: OR={R.loc['Намерен искать работу', 'OR']:.2f}; успеваемость, сон, демография до начала почти не различаются; AUC всех признаков около 0.6 — выбор плохо предсказуем.")

## 8. Сводка и экспорт

In [ ]:
R2 = pd.concat(RES2, ignore_index=True); R2["знач"] = R2.p.apply(lambda p: "" if pd.isna(p) else stars(p))
R2.round(4).to_csv("results_D.csv", index=False)
print("Строк в сводной таблице:", len(R2)); print(R2.groupby("блок").size().to_string())
keyn = {"Работающих 3в / 4в, %": (round(d.emp3.mean()*100, 1), round(d.emp4.mean()*100, 1)), "Работающих 4в (n)": len(w4),
        "Из них с профилем (n)": int(w4.profile.notna().sum()), "Неработавшие 3в с ответом в 4в": len(Bn), "Из них начали": int(Bn.start.sum()),
        "Панель (n)": int(d.panel.sum())}
print(); [print(f"{k}: {v}") for k, v in keyn.items()]
with zipfile.ZipFile("results_D.zip", "w") as z:
    for f in glob.glob("figs/D*.png") + ["results_D.csv"]: z.write(f)
try:
    from google.colab import files; files.download("results_D.zip")
except Exception: pass

## Скорректированный план презентации (10 слайдов) и откуда брать материал

> **Важная поправка к черновику плана.** В 4-й волне вопросы «опыт работы помогает усваивать материал / из-за работы страдает успеваемость / трудно совмещать» (`intern5_4w_*`) относятся к **работе**, а не к стажировке: в анкете 4-й волны формулировка «о вашей работе», и вопрос показывается всем, кто работает. В 3-й волне те же утверждения относились к **стажировке**. Поэтому оценка пользы и цены работы возможна (блок 3), а стажировки служат отдельной повторной проверкой (блок 7).

| № | Слайд | Содержание | Ячейки и рисунки |
|---|---|---|---|
| 1 | Титул | Цена, выгода и выбор: работа во время учёбы. Вывод в одну строку | — |
| 2 | Вопрос и обоснование | Формулировка D, 3 причины важности | — |
| 3 | Данные и выборка | Воронка, переходы, осыпание | `D00_funnel`, раздел 1b |
| 4 | Дизайн и переменные | Карта: цена / содержание и мотив / выбор + схема 3→4 | таблица гипотез в начале |
| 5 | Кто и как работает | 44% и 48% работают, типы, часы, поток 3→4 | `D01_who_works`, `D01b_alluvial` |
| 6 | Зачем и что за работа | Мотивы, соответствие специальности | `D01c_motives_match` |
| 7 | Цена | Бюджет времени, сон, привычки, учёба; две волны | `D01_time_budget`, `D02_cost_benefit_outcomes` |
| 8 | **Главный: польза и цена по профилям** | 2×2 «содержание × интенсивность», механизм конфликта | `D03_profiles_benefit_cost`, `D05_profiles_heatmap`, блоки 3–4 |
| 9 | Отбор до работы | Кто начал работать, чем отличались до начала, предсказуемость | `D04_road_to_work`, `D04b_love_plot` |
| 10 | Выводы и ограничения | 3 тезиса, E-value, осыпание, повтор на стажировках | раздел 7b, блок 7 |

**Тон формулировок.** Везде «связано», не «приводит». «Воспринимаемая польза», а не «польза». Бюджет времени — «планируют», а не «тратят». Малые группы (<60) помечайте и показывайте ДИ.